In [5]:
import pandas as pd
import numpy as np
import os
import json

def split(text):
    text = text.replace("• ", "")
    text = text.split("\n")
    
    res = ""
    for index, sub in enumerate(text):
        res += f"{index+1}. {sub} "

        
    return res

In [6]:
df_interest = pd.read_excel("./dataset/CABIN_info.xlsx")
df_interest["Illustrative Occupations"] = df_interest["Illustrative Occupations"].apply(split)
df_interest["Questionnaire Items"] = df_interest["Questionnaire Items"].apply(split)
df_occup = pd.read_excel("./dataset/Occ_info.xlsx")
df_occup["Task"] = df_occup["Task"].apply(split)
df_occup.tail()

,O*NET-SOC Code,Title,Description,Task
918,53-7071.00,Gas Compressor and Gas Pumping Station Operators,"Operate steam-, gas-, electric motor-, or inte...",1. Monitor meters and pressure gauges to deter...
919,53-7072.00,"Pump Operators, Except Wellhead Pumpers","Tend, control, or operate power-driven, statio...",1. Monitor gauges and flowmeters and inspect e...
920,53-7073.00,Wellhead Pumpers,Operate power pumps and auxiliary equipment to...,1. Monitor pumps and flow lines for gas and fl...
921,53-7081.00,Refuse and Recyclable Material Collectors,Collect and dump refuse or recyclable material...,1. Inspect trucks prior to beginning routes to...
922,53-7121.00,"Tank Car, Truck, and Ship Loaders","Load and unload chemicals and bulk solids, suc...","1. Seal outlet valves on tank cars, barges, an..."


In [30]:

cols = {
    "AI_model": [],
    "basic_interest": [],
    "occupation": [],
    "rating_raw": [],
}

for ai_model in ["gpt-4o", "claude-3-7-sonnet-latest", "deepseek-chat"]: # "gpt-4o", "claude-3-7-sonnet-latest"
    for interest_row in range(len(df_interest)):
        for occupation_row in range(len(df_occup)):
        # for occupation_row in range(2):
            try:
                with open(f"./src/logs/{ai_model}_{interest_row}_{occupation_row}.txt", "r") as file:
                    resp = file.readline()
            except Exception as e:
                print(f"{ai_model}_{interest_row}_{occupation_row}")
                resp = None

            cols["AI_model"].append(ai_model)
            cols["basic_interest"].append(df_interest.loc[interest_row, "Basic Interest"])
            cols["occupation"].append(df_occup.loc[occupation_row, "Title"])
            cols["rating_raw"].append(resp)


df_res = pd.DataFrame(cols)
df_res



,AI_model,basic_interest,occupation,rating_raw
0,gpt-4o,Life Science,Fast Food and Counter Workers,1
1,gpt-4o,Life Science,"Laborers and Freight, Stock, and Material Move...",1
2,gpt-4o,Life Science,Software Developers,1
3,gpt-4o,Life Science,Computer Systems Analysts,1
4,gpt-4o,Life Science,Graphic Designers,1
...,...,...,...,...
113524,deepseek-chat,Protective Service,Gas Compressor and Gas Pumping Station Operators,1
113525,deepseek-chat,Protective Service,"Pump Operators, Except Wellhead Pumpers",1
113526,deepseek-chat,Protective Service,Wellhead Pumpers,1
113527,deepseek-chat,Protective Service,Refuse and Recyclable Material Collectors,1


In [31]:
df_res.to_excel("./output/raw.xlsx", index=False)

In [32]:
df_gpt4o = df_res[df_res["AI_model"] == "gpt-4o"]
df_gpt4o = df_gpt4o[['basic_interest', 'occupation', 'rating_raw']]
df_gpt4o.columns = ['basic_interest', 'occupation', 'gpt-4o']
df_gpt4o

,basic_interest,occupation,gpt-4o
0,Life Science,Fast Food and Counter Workers,1
1,Life Science,"Laborers and Freight, Stock, and Material Move...",1
2,Life Science,Software Developers,1
3,Life Science,Computer Systems Analysts,1
4,Life Science,Graphic Designers,1
...,...,...,...
37838,Protective Service,Gas Compressor and Gas Pumping Station Operators,1
37839,Protective Service,"Pump Operators, Except Wellhead Pumpers",1
37840,Protective Service,Wellhead Pumpers,1
37841,Protective Service,Refuse and Recyclable Material Collectors,1


In [33]:
df_deepseek = df_res[df_res["AI_model"] == "deepseek-chat"]
df_deepseek = df_deepseek[['rating_raw']]
df_deepseek.columns = ['deepseek-chat']
df_deepseek = df_deepseek.reset_index(drop=True)
df_deepseek

,deepseek-chat
0,1
1,1
2,1
3,1
4,1
...,...
37838,1
37839,1
37840,1
37841,1


In [34]:
df_claude = df_res[df_res["AI_model"] == "claude-3-7-sonnet-latest"]
df_claude = df_claude[['rating_raw']]
df_claude.columns = [ 'claude-3-7-sonnet-latest']
df_claude = df_claude.reset_index(drop=True)
df_claude

,claude-3-7-sonnet-latest
0,1
1,1
2,1
3,1
4,1
...,...
37838,1
37839,1
37840,1
37841,1


In [35]:
df_merged = pd.concat([df_gpt4o, df_deepseek, df_claude], axis=1)
df_merged

,basic_interest,occupation,gpt-4o,deepseek-chat,claude-3-7-sonnet-latest
0,Life Science,Fast Food and Counter Workers,1,1,1
1,Life Science,"Laborers and Freight, Stock, and Material Move...",1,1,1
2,Life Science,Software Developers,1,1,1
3,Life Science,Computer Systems Analysts,1,1,1
4,Life Science,Graphic Designers,1,1,1
...,...,...,...,...,...
37838,Protective Service,Gas Compressor and Gas Pumping Station Operators,1,1,1
37839,Protective Service,"Pump Operators, Except Wellhead Pumpers",1,1,1
37840,Protective Service,Wellhead Pumpers,1,1,1
37841,Protective Service,Refuse and Recyclable Material Collectors,1,1,1


In [36]:
df_merged.to_excel("./output/merged.xlsx", index=False)